In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
import folium
import json
import time
import numpy as np
import h3
from folium.plugins import HeatMap
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
import time

In [3]:
sys.path.append('src')

In [42]:
from fetch_data import *
from data_io import *
from data_prep import *
from scoring import *
from clustering import *
from visualization import *

In [5]:
#define boundaries
ATLANTA_BBOX  = [33.64, -84.55, 33.89, -84.29]

In [6]:
all_pois = []
all_pois = query_restaurant_data(ATLANTA_BBOX, all_pois)
all_pois = query_park_data(ATLANTA_BBOX, all_pois)
all_pois = query_hospital_and_clinic_data(ATLANTA_BBOX, all_pois)
print(f"Total POIs fetched: {len(all_pois)}")

Fetching restaurant data
 Found 1029 restaurant
Fetching cafe data
 Found 185 cafe
Fetching park data
 Found 504 park
Fetching hospital data
 Found 18 hospital
Fetching clinic data
 Found 48 clinic
Total POIs fetched: 1784


In [7]:
name_of_the_file = "atlanta_pois"

In [ ]:
save_pois(all_pois, name_of_the_file)

Saved as GeoJSON
Saved as CSV


In [8]:
df_pois = load_pois(name_of_the_file)

Loaded 1784
Summary by type:
type
restaurant    1029
park           504
cafe           185
clinic          48
hospital        18
Name: count, dtype: int64


In [9]:
hexagons = create_hex_grids(df_pois)

Generated 1031 hexagons at resolution 8


In [10]:
hex_data = []

In [11]:
df_hexagons = calculate_accessibility_scores(hexagons, df_pois)

Calculating accessibility scores for each hexagon...
  Processing hexagon 0/1031...
  Processing hexagon 50/1031...
  Processing hexagon 100/1031...
  Processing hexagon 150/1031...
  Processing hexagon 200/1031...
  Processing hexagon 250/1031...
  Processing hexagon 300/1031...
  Processing hexagon 350/1031...
  Processing hexagon 400/1031...
  Processing hexagon 450/1031...
  Processing hexagon 500/1031...
  Processing hexagon 550/1031...
  Processing hexagon 600/1031...
  Processing hexagon 650/1031...
  Processing hexagon 700/1031...
  Processing hexagon 750/1031...
  Processing hexagon 800/1031...
  Processing hexagon 850/1031...
  Processing hexagon 900/1031...
  Processing hexagon 950/1031...
  Processing hexagon 1000/1031...

✓ Calculated accessibility scores for 1031 hexagons

Accessibility Score Statistics:
       restaurant_accessibility  park_accessibility  clinic_accessibility
count               1031.000000         1031.000000           1031.000000
mean                  

In [17]:
user_weights = {
    'restaurant': 0.5,
    'park': 0.1,
    'clinic': 0.4
}



In [13]:
df_hexagons = apply_user_weights(df_hexagons, user_weights)
print(df_hexagons.head())


Applying User Weights
User preferences: {'restaurant': 0.5, 'park': 0.1, 'clinic': 0.4}
Sum of weights: 1.00 (should be 1.0)

User Match Score Statistics:
count    1031.000000
mean        0.105213
std         0.137191
min         0.000000
25%         0.027221
50%         0.056185
75%         0.118947
max         0.943098
Name: user_match_score, dtype: float64
            hex_id        lat        lon  restaurant_accessibility  \
0  8844c1a069fffff  33.673412 -84.523988                  0.000000   
1  8844c1aa21fffff  33.690366 -84.381513                  0.000000   
2  8844c1b8b7fffff  33.665531 -84.430036                  4.396577   
3  8844c1ab19fffff  33.688773 -84.362743                  0.000000   
4  8844c1a869fffff  33.709967 -84.356853                  0.431018   

   park_accessibility  clinic_accessibility  restaurant_norm  park_norm  \
0            1.444768              0.538529         0.000000   0.071015   
1            2.067756              0.191144         0.000000   0.1

In [28]:


df_hexagons_custom = cluster_based_on_score(df_hexagons)


Thresholds: High = 0.090, Medium = 0.036

Suitability Distribution:
suitability_label
Okay             351
Less Suitable    340
Most Suitable    340
Name: count, dtype: int64

SUITABILITY TIER CHARACTERISTICS

Most Suitable (340 hexagons):
  Match Score Range: 0.090 - 0.943
  Avg restaurant Access: 9.872
  Avg park Access: 6.804
  Avg clinic Access: 2.361

Okay (351 hexagons):
  Match Score Range: 0.036 - 0.090
  Avg restaurant Access: 1.287
  Avg park Access: 2.686
  Avg clinic Access: 0.638

Less Suitable (340 hexagons):
  Match Score Range: 0.000 - 0.036
  Avg restaurant Access: 0.222
  Avg park Access: 1.249
  Avg clinic Access: 0.160


In [37]:
df_hexagons_custom.head()

,hex_id,lat,lon,restaurant_accessibility,park_accessibility,clinic_accessibility,restaurant_norm,park_norm,clinic_norm,user_match_score,suitability,suitability_label
0,8844c1a069fffff,33.673412,-84.523988,0.000000,1.444768,0.538529,0.000000,0.071015,0.073387,0.036456,1,Okay
1,8844c1aa21fffff,33.690366,-84.381513,0.000000,2.067756,0.191144,0.000000,0.101636,0.026048,0.020583,2,Less Suitable
2,8844c1b8b7fffff,33.665531,-84.430036,4.396577,4.379221,0.745595,0.071216,0.215251,0.101605,0.097775,0,Most Suitable
3,8844c1ab19fffff,33.688773,-84.362743,0.000000,1.644581,0.000000,0.000000,0.080836,0.000000,0.008084,2,Less Suitable
4,8844c1a869fffff,33.709967,-84.356853,0.431018,2.933234,0.523418,0.006982,0.144177,0.071328,0.046440,1,Okay


In [34]:
folium_map = create_suitability_map(
    df_hexagons_custom,
    user_weights)

Suitability distribution:
suitability
0    340
1    351
2    340
Name: count, dtype: int64
Color mapping for 3 clusters: {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
Adding hexagons to map...
  Added 0/1031 hexagons...
  Added 50/1031 hexagons...
  Added 100/1031 hexagons...
  Added 150/1031 hexagons...
  Added 200/1031 hexagons...
  Added 250/1031 hexagons...
  Added 300/1031 hexagons...
  Added 350/1031 hexagons...
  Added 400/1031 hexagons...
  Added 450/1031 hexagons...
  Added 500/1031 hexagons...
  Added 550/1031 hexagons...
  Added 600/1031 hexagons...
  Added 650/1031 hexagons...
  Added 700/1031 hexagons...
  Added 750/1031 hexagons...
  Added 800/1031 hexagons...
  Added 850/1031 hexagons...
  Added 900/1031 hexagons...
  Added 950/1031 hexagons...
  Added 1000/1031 hexagons...


In [36]:
save_map(folium_map, output_file='atlanta_suitability_map.html')

Saved: data/output_data/atlanta_suitability_map.html


In [41]:
df_hexagons_custom.columns

Index(['hex_id', 'lat', 'lon', 'restaurant_accessibility',
       'park_accessibility', 'clinic_accessibility', 'restaurant_norm',
       'park_norm', 'clinic_norm', 'user_match_score', 'suitability',
       'suitability_label'],
      dtype='object')

In [43]:
save_csv(df_hexagons_custom, 'atlanta_suitability')
save_geojson(df_hexagons_custom, 'atlanta_suitability')

Saved CSV: data/output_data/atlanta_suitability.csv
Saved GeoJSON: data/output_data/atlanta_suitability.geojson
